# Data Analysis and Visualisation

## Lab 4: Gaps and merge

### Preliminaries

In [ ]:
import pandas as pd

### Part 1

#### Task 1: Load the loans and look for gaps

In [ ]:
loans = pd.read_csv("data/library_loans_clean.csv", parse_dates=["loan_date", "return_date"])

print(loans.shape)
print(loans.dtypes)
print(loans.isna().sum())
print((loans.isna().mean() * 100).round(1))

(600, 8)
loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object
loan_id         0
member_id       0
title           0
genre           0
loan_date       3
return_date    36
late_fee        0
returned        0
dtype: int64
loan_id        0.0
member_id      0.0
title          0.0
genre          0.0
loan_date      0.5
return_date    6.0
late_fee       0.0
returned       0.0
dtype: float64


Only two columns have gaps: loan_date (3 missing, 0.5 %) and return_date (36 missing, 6.0 %).

#### Task 2: Two kinds of missing

In [ ]:
display(loans[loans["loan_date"].isna()])

print((loans["return_date"].isna() & loans["returned"]).sum())
print((loans["return_date"].isna() & ~loans["returned"]).sum())

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
462,L-30463,M0007,Blue Moon Protocol,Sci-fi,NaT,2026-05-24,0.0,True
467,L-30468,M0018,Blue Moon Protocol,Sci-fi,NaT,2026-06-08,0.0,True
571,L-30572,M0090,Mimmi and the Moon,Children,NaT,2026-06-26,0.0,True


6
30


30 loans have no return date and the book is not back: missing for a reason, nobody has returned it yet. 6 have no return date but the book is back: missing by accident, the return date should exist and was not recorded.

In [ ]:
loans["has_return_date"] = loans["return_date"].notna()
print(loans["has_return_date"].value_counts())

has_return_date
True     564
False     36
Name: count, dtype: int64


#### Task 3: Look at the members

In [ ]:
members_raw = pd.read_csv("data/library_members.csv")
members = members_raw.copy()

print(members.shape)
members.info()
print(members["birth_year"].describe())

(148, 3)
<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.6 KB
count     130.000000
mean     1983.384615
std        25.158223
min      1900.000000
25%      1978.250000
50%      1988.000000
75%      1996.750000
max      2030.000000
Name: birth_year, dtype: float64


In [ ]:
print("Wrong birth year:", "1900:", (members["birth_year"] == 1900).sum(), ", 2030:", (members["birth_year"] == 2030).sum())
print("Duplicates:", members.duplicated().sum(), members.duplicated(subset="member_id").sum())

Wrong birth year: 1900: 8 , 2030: 3
Duplicates: 4 4


Problems: `branch` has 12 missing, `birth_year` has 18 missing, the minimum 1900 (8 members) looks like a placeholder, the maximum 2030 (3 members) is in the future, and 4 members are in the table twice (4 duplicate rows and 4 duplicate `member_ids`).

#### Task 4: Placeholders and impossible values

In [ ]:
members["birth_year"] = members["birth_year"].where(members["birth_year"].between(1920, 2015))

print(members["birth_year"].isna().sum())
print(members["birth_year"].describe())

29
count     119.000000
mean     1987.815126
std        12.062666
min      1957.000000
25%      1979.000000
50%      1988.000000
75%      1996.500000
max      2010.000000
Name: birth_year, dtype: float64


1900 and 2030 are not real birth years, so they became `NaN`. Now 29 are missing (18 + 8 + 3). The minimum is 1957 and the maximum 2010, and the mean rose from 1983.38 to 1987.82.

#### Task 5: Remove, fill, or leave

In [ ]:
print(members.dropna().shape)
print(members.dropna(subset=["branch"]).shape)
print(members.dropna(subset=["birth_year"]).shape)

(107, 3)
(136, 3)
(119, 3)


In [ ]:
members["branch"] = members["branch"].fillna("Unknown")
print(members["branch"].value_counts())

branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64


In [ ]:
birthyear_filled = members["birth_year"].fillna(members["birth_year"].median())
print(members["birth_year"].mean(), birthyear_filled.mean())
print(members["birth_year"].std(), birthyear_filled.std())

1987.81512605042 1987.8513513513512
12.062665744223288 10.807760417230606


Using `dropna()` would lose 41 of 148 members, so we do not remove rows. The missing branch becomes "Unknown" so those members stay in the counts. Filling `birth_year` with the median keeps the mean almost the same (1987.82 vs 1987.85) but makes the spread smaller (12.06 vs 10.81), because 29 members would get the same invented value. So we leave `birth_year` as `NaN`.

#### Task 6: Merge, and check

In [ ]:
print(f"Loans table shape: {loans.shape}")
print(f"Members table shape: {members.shape}")

Loans table shape: (600, 9)
Members table shape: (148, 3)


In [ ]:
merged = loans.merge(members, on="member_id", how="left")
print(merged.shape)
print("Late fees before merge:", loans["late_fee"].sum())
print("Late fees after merge:", merged["late_fee"].sum())

(616, 11)
Late fees before merge: 7487.5
Late fees after merge: 7725.0


In [ ]:
loans.merge(members, on="member_id", how="left", validate="many_to_one")

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 member_id
    M0015
    M0033
    M0065
    M0071 ...

#### Task 7: Fix the key, and find out who is missing

In [ ]:
members = members.drop_duplicates().reset_index(drop=True)
print("Members table shape without duplicates:", members.shape)

merged = loans.merge(members, on="member_id", how="left", validate="many_to_one", indicator=True)
print("The shape of the merged table:", merged.shape)
print("Late fees after merge:", merged["late_fee"].sum())

Members table shape without duplicates: (144, 3)
The shape of the merged table: (600, 12)
Late fees after merge: 7487.5


In [ ]:
left_only = merged[merged["_merge"] == "left_only"]
print("Number of loans with missing members:", len(left_only))
print("Members missing from the members table:", left_only["member_id"].nunique())
print("Late fees of the missing members:", left_only["late_fee"].sum())

Number of loans with missing members: 31
Members missing from the members table: 6
Late fees of the missing members: 347.5


In [ ]:
inner = loans.merge(members, on="member_id", how="inner")
print(inner.shape, inner["late_fee"].sum())

(569, 11) 7140.0


After removing the 4 duplicates there are 144 members, and the merge gives 600 rows and 7487.5 kr again. 31 loans from 6 members have no member record (347.5 kr). An inner merge would have dropped them and given 569 rows and 7140.0 kr.

#### Task 8: Use the merged table

In [ ]:
merged["branch"] = merged["branch"].fillna("No member record")
print(merged["branch"].value_counts())

branch
Central             144
South               130
North               126
East                 72
Harbour              51
Unknown              46
No member record     31
Name: count, dtype: int64


Central has the most loans (144).

"Unknown" is a member who exists but has no branch recorded, while "No member record" is a loan whose member is not in the member table at all.